# YOLO Dataset Visualisation

Universal notebook for inspecting any of the four YOLO tile-preparation
datasets (detection RGB, detection IRRG, segmentation RGB, segmentation IRRG).

**Usage**
1. Set `SPLITS_DIR` to the root of the dataset you want to inspect.
2. Set `TASK` to `'detection'` or `'segmentation'`.
3. Set `CLASS_NAMES` to match the class IDs used in your label files.
4. Run all cells.

**Figures produced**
- Annotated preview of a single patch (bounding boxes or polygon outlines)
- Bar chart of positive (that include the target class) vs negative images per split (include backgorund areas without the target class)
- Per-class annotation counts across all three splits


**Import libraries**

In [ ]:
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

# Locate project root
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Enable imports from src
sys.path.append(str(PROJECT_ROOT))

from src.paths import RAW_DIR, EXTERNAL_DIR, INTERIM_DIR, PROCESSED_DIR


**Configuration — edit these paths before running**

In [ ]:
# USER CONFIGURATION
# Point SPLITS_DIR at the root of whichever dataset you want to inspect:
#
#   Detection RGB  →  PROCESSED_DIR / "splits_rgb"
#   Detection IRRG →  PROCESSED_DIR / "splits_irrg"
#   Segmentation RGB  →  PROCESSED_DIR / "splits_rgb_seg"
#   Segmentation IRRG →  PROCESSED_DIR / "splits_irrg_seg"
#
# Set TASK to "detection" or "segmentation" so the patch-preview function
# draws bounding boxes or polygon outlines respectively.
#
# Set CLASS_NAMES to the ordered list of class names that matches the integer
# IDs in your label files.  For segmentation datasets with a single 'Tree'
# class this is simply ['Tree'].

SPLITS_DIR   = PROCESSED_DIR /"splits_rgb" # change the split foldersbased on preferences 
TASK         = "detection"           # "detection" or "segmentation"
CLASS_NAMES  = ["Alnus", "Betula", "Fraxinus", "Platanus", "Quercus"]   

**Patch preview — detection (bounding boxes)**

In [ ]:
def preview_detection_patch(
    image_path: str,
    label_path: str,
    class_names: list,
    save_path: str | None = None,
    show: bool = True,
) -> None:
    """
    Display one detection patch with its YOLO bounding-box annotations overlaid.

    Bounding boxes are drawn in red with the class name printed in yellow above
    each box.  If no label file exists the patch is shown as a background image.

    Args:
        image_path:  Path to the GeoTIFF image patch.
        label_path:  Path to the YOLO .txt annotation file.
        class_names: Ordered list of class name strings.
        save_path:   If provided the figure is saved here (PNG).
        show:        Whether to call plt.show().
    """
    with rasterio.open(image_path) as src:
        img_data = src.read()

    img = np.transpose(img_data, (1, 2, 0))
    if img.dtype != np.uint8:
        img = img / img.max()
    height, width = img.shape[:2]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img)
    ax.set_title(os.path.basename(image_path))

    if not os.path.exists(label_path):
        ax.set_xlabel("No label file (background patch)")
        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
        if show:
            plt.show()
        plt.close(fig)
        return

    with open(label_path) as f:
        lines = f.readlines()

    for line in lines:
        try:
            parts = line.strip().split()
            class_id = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])

            box_w = bw * width
            box_h = bh * height
            xmin  = xc * width  - box_w / 2
            ymin  = yc * height - box_h / 2

            rect = mpatches.Rectangle(
                (xmin, ymin), box_w, box_h,
                linewidth=2, edgecolor="red", facecolor="none",
            )
            ax.add_patch(rect)

            label = class_names[class_id] if class_id < len(class_names) else str(class_id)
            ax.text(
                xmin, ymin - 3, label,
                color="yellow", fontsize=8,
                bbox=dict(facecolor="black", alpha=0.5, pad=1),
            )
        except Exception as e:
            print(f"Skipping malformed line: {line.strip()} ({e})")

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


**Patch preview — segmentation (polygon outlines)**

In [ ]:
def preview_segmentation_patch(
    image_path: str,
    label_path: str,
    class_names: list,
    save_path: str | None = None,
    show: bool = True,
) -> None:
    """
    Display one segmentation patch with its YOLO polygon annotations overlaid.

    Each crown polygon is drawn as a green outline with the class name centred
    inside.  If no label file exists the patch is shown as a background image.

    Args:
        image_path:  Path to the GeoTIFF image patch.
        label_path:  Path to the YOLO segmentation .txt annotation file.
        class_names: Ordered list of class name strings.
        save_path:   If provided the figure is saved here (PNG).
        show:        Whether to call plt.show().
    """
    with rasterio.open(image_path) as src:
        img_data = src.read()

    img = np.transpose(img_data, (1, 2, 0))
    if img.dtype != np.uint8:
        img = img / img.max()
    height, width = img.shape[:2]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img)
    ax.set_title(os.path.basename(image_path))
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_aspect("equal")

    if not os.path.exists(label_path):
        ax.set_xlabel("No label file (background patch)")
        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
        if show:
            plt.show()
        plt.close(fig)
        return

    with open(label_path) as f:
        lines = f.readlines()

    for line in lines:
        parts    = line.strip().split()
        class_id = int(parts[0])
        coords   = list(map(float, parts[1:]))
        xs = [coords[i]     * width  for i in range(0, len(coords), 2)]
        ys = [coords[i + 1] * height for i in range(0, len(coords) - 1, 2)]
        xs.append(xs[0])
        ys.append(ys[0])

        ax.plot(xs, ys, color="green", linewidth=2)
        label = class_names[class_id] if class_id < len(class_names) else str(class_id)
        ax.text(
            sum(xs) / len(xs), sum(ys) / len(ys), label,
            color="white", fontsize=9, ha="center", va="center",
            bbox=dict(facecolor="black", alpha=0.5, pad=1, edgecolor="none"),
        )

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


**Preview one patch**

In [ ]:
# Edit the paths below to preview a specific patch
split      = "train"
patch_stem = "RGB_34FN2_patch_11_45"   # filename without extension

image_path = str(SPLITS_DIR / split / "images" / f"{patch_stem}.tif")
label_path = str(SPLITS_DIR / split / "labels" / f"{patch_stem}.txt")
save_path  = str(SPLITS_DIR / split / "previews" / f"{patch_stem}_preview.png")

# Create the previews directory if it does not yet exist
os.makedirs(str(SPLITS_DIR / split / "previews"), exist_ok=True)

if TASK == "detection":
    preview_detection_patch(image_path, label_path, CLASS_NAMES, save_path, show=True)
else:
    preview_segmentation_patch(image_path, label_path, CLASS_NAMES, save_path, show=True)


**Images per split (positive vs negative)**

In [ ]:
def count_images_per_split(splits_dir: Path) -> pd.DataFrame:
    """
    Count positive (annotated) and negative (background) patches per split.

    A patch is considered positive if a label file with the same stem exists in
    the labels/ sub-directory; otherwise it is background.

    Args:
        splits_dir: Root directory containing train/, val/, and test/ sub-dirs.

    Returns:
        DataFrame with columns ['split', 'positive', 'negative', 'total'].
    """
    rows = []
    for split in ("train", "val", "test"):
        img_dir = splits_dir / split / "images"
        lbl_dir = splits_dir / split / "labels"

        img_stems = {
            Path(f).stem for f in os.listdir(img_dir)
            if f.lower().endswith(".tif")
        }
        lbl_stems = {
            Path(f).stem for f in os.listdir(lbl_dir)
            if f.lower().endswith(".txt")
        }

        rows.append({
            "split":    split,
            "positive": len(img_stems & lbl_stems),
            "negative": len(img_stems - lbl_stems),
            "total":    len(img_stems),
        })
    return pd.DataFrame(rows)


df_images = count_images_per_split(SPLITS_DIR)
print(df_images.to_string(index=False))

ax = df_images.plot.bar(x="split", rot=0, figsize=(7, 4))
plt.title("Images per split")
plt.xlabel("Dataset split")
plt.ylabel("Number of images")
plt.legend(title="Image type", loc="upper right")
plt.tight_layout()
plt.show()


**Class annotations per split**

In [ ]:
def count_labels(label_dir: str, classes: list) -> Counter:
    """
    Read all YOLO label files in a directory and count annotations per class.

    The first token on each line is the integer class ID.

    Args:
        label_dir: Path to the labels/ sub-directory for one split.
        classes:   Ordered list of class name strings.

    Returns:
        collections.Counter mapping class name → annotation count.
    """
    counter: Counter = Counter()
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(label_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                if 0 <= cls_id < len(classes):
                    counter[classes[cls_id]] += 1
    return counter


split_counters = {}
for split in ("train", "val", "test"):
    lbl_dir = str(SPLITS_DIR / split / "labels")
    split_counters[split] = count_labels(lbl_dir, CLASS_NAMES)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=120, sharey=True)
colors = plt.cm.Set2(np.linspace(0, 1, len(CLASS_NAMES)))

for ax, split in zip(axes, ("train", "val", "test")):
    counts = [split_counters[split].get(c, 0) for c in CLASS_NAMES]
    bars   = ax.bar(CLASS_NAMES, counts, color=colors, edgecolor="black")
    ax.set_title(split.upper(), fontsize=14, weight="bold")
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=10)
    for bar, count in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            str(count),
            ha="center", va="bottom", fontsize=9, weight="bold",
        )

fig.suptitle("Class annotations per split", fontsize=16, weight="bold", y=1.02)
plt.tight_layout()
plt.show()
